In [ ]:
import torch
import numpy as np
import json
import argparse
from pathlib import Path
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import Dict, List, Tuple
from scipy import stats
from scipy.stats import ttest_ind, ttest_rel
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# Dataset Loading
# ============================================================================

def load_prolific_dataset(filepath: str, max_samples: int = None):
    """Load ProlificAI dataset from JSONL."""
    print(f"\nLoading ProlificAI from {filepath}...")
    
    from pathlib import Path
    if not Path(filepath).exists():
        print(f"  ✗ File not found: {filepath}")
        return []
    
    examples = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            if max_samples and len(examples) >= max_samples:
                break
            
            line = line.strip()
            if not line:
                continue
            
            try:
                item = json.loads(line)
                question = item.get('question', '').strip()
                chosen = item.get('chosen', '').strip()
                
                if question and chosen:
                    examples.append({
                        'prompt': question,
                        'response': chosen,
                        'source': 'ProlificAI'
                    })
            except json.JSONDecodeError:
                continue
    
    print(f"  ✓ Loaded {len(examples)} examples")
    return examples

In [ ]:
# ============================================================================
# Full-Sequence Loss Calculation
# ============================================================================
def compute_loss(model, tokenizer, prompt, response):
    """
    Tokenize together, then find actual boundary.
    """
    
    model.eval()
    device = model.device
    
    # Full text: prompt + response
    full_text = prompt + " " + response
    
    # Tokenize
    inputs = tokenizer(full_text, return_tensors='pt', truncation=True, max_length=512).to(device)
    prompt_inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(device)
    
    # Compute loss only on response tokens (not prompt)
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs['input_ids'])
        
        # Total loss
        return outputs.loss.item()

# ============================================================================
# Proper Statistical Analysis
# ============================================================================

def compute_statistics_on_examples(
    model,
    tokenizer,
    examples: List[Dict],
    batch_size: int = 4
) -> Dict:
    """
    Compute loss on examples with proper statistical tracking.
    
    Returns:
        Dict with:
        - per-example losses (for statistical tests)
        - mean/std/CI
        - per-dataset breakdowns
        - perplexity (standard metric)
    """
    
    model.eval()
    
    all_losses = []
    per_source_losses = {'ProlificAI': []}
    
    # Compute loss for each example
    for example in tqdm(examples, desc="Computing losses"):
        prompt = example['prompt']
        response = example['response']
        source = example['source']
        
        # Use response-only loss
        loss = compute_loss(model, tokenizer, prompt, response)
        
        all_losses.append(loss)
        per_source_losses[source].append(loss)
    
    # Compute statistics
    losses_array = np.array(all_losses)
    
    # Overall statistics
    mean_loss = np.mean(losses_array)
    std_loss = np.std(losses_array, ddof=1)  # Sample std
    sem_loss = stats.sem(losses_array)  # Standard error of mean
    
    # 95% confidence interval
    ci_95 = stats.t.interval(
        0.95,
        len(losses_array) - 1,
        loc=mean_loss,
        scale=sem_loss
    )
    
    # Perplexity (standard metric - NOT arbitrary!)
    # Perplexity = exp(cross_entropy) is the standard definition in NLP
    # Interpretation: "On average, model is as confused as if choosing from N options"
    perplexity = np.exp(mean_loss)
    
    # Per-dataset statistics
    per_source_stats = {}
    for source, losses in per_source_losses.items():
        if losses:
            losses_arr = np.array(losses)
            per_source_stats[source] = {
                'n_examples': len(losses),
                'mean_loss': float(np.mean(losses_arr)),
                'std_loss': float(np.std(losses_arr, ddof=1)),
                'perplexity': float(np.exp(np.mean(losses_arr))),
                'ci_95_lower': float(np.mean(losses_arr) - 1.96 * np.std(losses_arr) / np.sqrt(len(losses_arr))),
                'ci_95_upper': float(np.mean(losses_arr) + 1.96 * np.std(losses_arr) / np.sqrt(len(losses_arr))),
            }
    
    return {
        # Per-example data (for statistical tests)
        'losses': losses_array.tolist(),
        
        # Summary statistics
        'n_examples': len(all_losses),
        'mean_loss': float(mean_loss),
        'std_loss': float(std_loss),
        'sem_loss': float(sem_loss),
        'ci_95_lower': float(ci_95[0]),
        'ci_95_upper': float(ci_95[1]),
        
        # Standard metrics (NOT arbitrary!)
        'perplexity': float(perplexity),
        
        # Per-dataset breakdown
        'per_source': per_source_stats,
    }

# ============================================================================
# Statistical Significance Testing
# ============================================================================

def compute_pairwise_statistics(
    results_by_model: Dict[str, Dict]
) -> Dict:
    """
    Compute statistical significance of pairwise differences.
    
    Uses:
    - Independent t-test (if different examples)
    - Paired t-test (if same examples - our case)
    - Effect size (Cohen's d)
    - Bootstrap confidence intervals
    """
    
    comparisons = {}
    model_names = list(results_by_model.keys())
    
    for i, model1 in enumerate(model_names):
        for model2 in model_names[i+1:]:
            
            losses1 = np.array(results_by_model[model1]['losses'])
            losses2 = np.array(results_by_model[model2]['losses'])
            
            # Check if same length (paired test possible)
            if len(losses1) == len(losses2):
                # Paired t-test (same prompts evaluated by both models)
                t_stat, p_value = ttest_rel(losses1, losses2)
                test_type = "paired_ttest"
            else:
                # Independent t-test
                t_stat, p_value = ttest_ind(losses1, losses2)
                test_type = "independent_ttest"
            
            # Effect size (Cohen's d)
            pooled_std = np.sqrt((np.var(losses1, ddof=1) + np.var(losses2, ddof=1)) / 2)
            cohens_d = (np.mean(losses1) - np.mean(losses2)) / pooled_std
            
            # Interpretation of effect size
            if abs(cohens_d) < 0.2:
                effect_interpretation = "negligible"
            elif abs(cohens_d) < 0.5:
                effect_interpretation = "small"
            elif abs(cohens_d) < 0.8:
                effect_interpretation = "medium"
            else:
                effect_interpretation = "large"
            
            # Bootstrap confidence interval for difference in means
            n_bootstrap = 1000
            diff_means = []
            for _ in range(n_bootstrap):
                sample1 = np.random.choice(losses1, size=len(losses1), replace=True)
                sample2 = np.random.choice(losses2, size=len(losses2), replace=True)
                diff_means.append(np.mean(sample1) - np.mean(sample2))
            
            bootstrap_ci = np.percentile(diff_means, [2.5, 97.5])
            
            # Determine winner (lower loss is better)
            mean1 = np.mean(losses1)
            mean2 = np.mean(losses2)
            
            if mean1 < mean2:
                winner = model1.split('/')[-1]
                advantage_pct = ((mean2 - mean1) / mean2) * 100
            else:
                winner = model2.split('/')[-1]
                advantage_pct = ((mean1 - mean2) / mean1) * 100
            
            comp_key = f"{model1.split('/')[-1]}_vs_{model2.split('/')[-1]}"
            
            comparisons[comp_key] = {
                'model1': model1.split('/')[-1],
                'model2': model2.split('/')[-1],
                'mean_loss_diff': float(mean1 - mean2),
                't_statistic': float(t_stat),
                'p_value': float(p_value),
                'significant_at_05': p_value < 0.05,
                'significant_at_01': p_value < 0.01,
                'cohens_d': float(cohens_d),
                'effect_size': effect_interpretation,
                'bootstrap_ci_95': [float(bootstrap_ci[0]), float(bootstrap_ci[1])],
                'winner': winner,
                'advantage_percent': float(advantage_pct),
                'test_type': test_type
            }
    
    return comparisons

# ============================================================================
# Power Analysis
# ============================================================================

def compute_required_sample_size(
    effect_size: float = 0.2,
    alpha: float = 0.05,
    power: float = 0.80
) -> int:
    """
    Compute required sample size for detecting effect.
    
    Args:
        effect_size: Cohen's d (0.5 = medium effect)
        alpha: Significance level (0.05 = 5%)
        power: Statistical power (0.80 = 80%)
    
    Returns:
        Required N per group
    
    Formula from Cohen (1988):
    n ≈ (Z_α/2 + Z_β)² × (2σ²) / d²
    """
    from scipy.stats import norm
    
    z_alpha = norm.ppf(1 - alpha/2)  # Two-tailed
    z_beta = norm.ppf(power)
    
    # Simplified formula assuming equal variances
    n = ((z_alpha + z_beta) ** 2) * 2 / (effect_size ** 2)
    
    return int(np.ceil(n))

# ============================================================================
# Visualization
# ============================================================================

def generate_visualizations(
    results: Dict,
    output_dir: str = "."
):
    """
    Generate publication-quality visualizations.
    
    Creates:
    1. Heatmap of pairwise comparisons
    2. Loss distribution plots
    3. Confidence interval comparison
    """
    
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        from matplotlib.patches import Rectangle
    except ImportError:
        print("\n⚠️  Matplotlib/seaborn not installed. Skipping visualizations.")
        print("   Install with: pip install matplotlib seaborn")
        return
    
    output_dir = Path(output_dir)
    
    # Extract data
    models = list(results['model_results'].keys())
    model_names = [m.split('/')[-1] for m in models]
    
    # 1. Heatmap of pairwise p-values
    n_models = len(models)
    p_matrix = np.ones((n_models, n_models))
    
    comparisons = results.get('pairwise_statistics', {})
    for comp_name, comp_data in comparisons.items():
        m1_name = comp_data['model1']
        m2_name = comp_data['model2']
        
        i = model_names.index(m1_name)
        j = model_names.index(m2_name)
        
        p_matrix[i, j] = comp_data['p_value']
        p_matrix[j, i] = comp_data['p_value']
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        p_matrix,
        annot=True,
        fmt='.4f',
        cmap='RdYlGn_r',
        xticklabels=model_names,
        yticklabels=model_names,
        vmin=0,
        vmax=0.1,
        cbar_kws={'label': 'p-value'}
    )
    plt.title('Statistical Significance of Pairwise Differences\n(Lower p-value = More significant difference)')
    plt.tight_layout()
    plt.savefig(output_dir / 'pairwise_significance_heatmap.png', dpi=300, bbox_inches='tight')
    print(f"\n✓ Saved: {output_dir / 'pairwise_significance_heatmap.png'}")
    plt.close()
    
    # 2. Mean loss with confidence intervals
    means = [results['model_results'][m]['mean_loss'] for m in models]
    ci_lowers = [results['model_results'][m]['ci_95_lower'] for m in models]
    ci_uppers = [results['model_results'][m]['ci_95_upper'] for m in models]
    
    errors_lower = [means[i] - ci_lowers[i] for i in range(n_models)]
    errors_upper = [ci_uppers[i] - means[i] for i in range(n_models)]
    
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(model_names))
    ax.bar(x, means, yerr=[errors_lower, errors_upper], capsize=5, alpha=0.7)
    ax.set_xlabel('Model')
    ax.set_ylabel('Mean Loss (lower is better)')
    ax.set_title('Social Reasoning Loss by Model (95% CI)')
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=45, ha='right')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_dir / 'loss_comparison_with_ci.png', dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {output_dir / 'loss_comparison_with_ci.png'}")
    plt.close()
    
    # 3. Effect size heatmap
    effect_matrix = np.zeros((n_models, n_models))
    
    for comp_name, comp_data in comparisons.items():
        m1_name = comp_data['model1']
        m2_name = comp_data['model2']
        
        i = model_names.index(m1_name)
        j = model_names.index(m2_name)
        
        effect_matrix[i, j] = comp_data['cohens_d']
        effect_matrix[j, i] = -comp_data['cohens_d']  # Negative for reverse comparison
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        effect_matrix,
        annot=True,
        fmt='.3f',
        cmap='RdBu_r',
        xticklabels=model_names,
        yticklabels=model_names,
        center=0,
        vmin=-1,
        vmax=1,
        cbar_kws={'label': "Cohen's d"}
    )
    plt.title("Effect Sizes of Pairwise Comparisons\n(Positive = Row model better, Negative = Column model better)")
    plt.tight_layout()
    plt.savefig(output_dir / 'effect_size_heatmap.png', dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {output_dir / 'effect_size_heatmap.png'}")
    plt.close()

# ============================================================================
# Model Evaluation
# ============================================================================

def evaluate_model(
    model_path: str,
    examples: List[Dict],
    batch_size: int = 4
) -> Dict:
    """Evaluate single model."""
    
    print(f"\n{'='*80}")
    print(f"Evaluating: {model_path}")
    print('='*80)
    
    # Load model
    print("Loading model...")
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Compute statistics
    results = compute_statistics_on_examples(model, tokenizer, examples, batch_size)
    
    print(f"\nResults:")
    print(f"  Mean Loss:     {results['mean_loss']:.4f} ± {results['std_loss']:.4f}")
    print(f"  95% CI:        [{results['ci_95_lower']:.4f}, {results['ci_95_upper']:.4f}]")
    print(f"  Perplexity:    {results['perplexity']:.2f}")
    
    if results['per_source']:
        print(f"\n  By Dataset:")
        for source, stats in results['per_source'].items():
            print(f"    {source}:")
            print(f"      Loss:  {stats['mean_loss']:.4f} ± {stats['std_loss']:.4f}")
            print(f"      PPL:   {stats['perplexity']:.2f}")
            print(f"      95% CI: [{stats['ci_95_lower']:.4f}, {stats['ci_95_upper']:.4f}]")
    
    # Clean up
    del model
    torch.cuda.empty_cache()
    
    return results

In [ ]:
# ============================================================================
# Main
# ============================================================================

_max_samples=None
_visualize=True
import os
# Stable result keys are based on the checkpoint labels used by the scorer.
_models=["./Qwen3-8B", "./Qwen3-8B-Base", "./llama-3.1-8b-base", "./llama-3.1-8B-instruct"]
_model_paths = {
    "./Qwen3-8B": os.environ.get("QWEN3_INSTRUCT_PATH", "./Qwen3-8B"),
    "./Qwen3-8B-Base": os.environ.get("QWEN3_BASE_PATH", "./Qwen3-8B-Base"),
    "./llama-3.1-8b-base": os.environ.get("LLAMA31_BASE_PATH", "./llama-3.1-8b-base"),
    "./llama-3.1-8B-instruct": os.environ.get("LLAMA31_INSTRUCT_PATH", "./llama-3.1-8B-instruct"),
}
_output="results/social_reasoning/full_sequence_baseline/baseline_results.json"
_max_samples=None
_batch_size=4
_prolific_file="data/social_reasoning/ProlificAI.jsonl"
_skip_prolific=False

print("="*80)
print("SOCIAL REASONING EVALUATION")
print("="*80)

# Power analysis for sample size
required_n = compute_required_sample_size(effect_size=0.2, power=0.80)
print(f"\nPower Analysis:")
print(f"  Required N per dataset: {required_n} (for 80% power, medium effect)")

if _max_samples is None:
    _max_samples = required_n
    print(f"  Using N={required_n} samples per dataset")
else:
    print(f"  Using user-specified N={_max_samples}")

# Load datasets
examples = []

if not _skip_prolific:
    prolific_examples = load_prolific_dataset(_prolific_file, _max_samples)
    examples.extend(prolific_examples)

if not examples:
    print("\nERROR: No examples loaded")

print(f"\nTotal examples: {len(examples)}")
print(f"  ProlificAI: {sum(1 for e in examples if e['source'] == 'ProlificAI')}")

# Evaluate models
all_results = {
    'metadata': {
        'n_examples_total': len(examples),
        'max_samples_per_dataset': _max_samples,
        'power_analysis': {
            'target_power': 0.80,
            'effect_size': 0.2,
            'required_n': required_n
        }
    },
    'model_results': {}
}

for model_path in _models:
    try:
        results = evaluate_model(_model_paths[model_path], examples, _batch_size)
        all_results['model_results'][model_path] = results
    except Exception as e:
        print(f"\nERROR evaluating {model_path}: {e}")
        continue

# Statistical comparisons
print("\n" + "="*80)
print("STATISTICAL COMPARISONS")
print("="*80)

pairwise_stats = compute_pairwise_statistics(all_results['model_results'])
all_results['pairwise_statistics'] = pairwise_stats

for comp_name, comp_data in pairwise_stats.items():
    print(f"\n{comp_name}:")
    print(f"  Δ Loss: {comp_data['mean_loss_diff']:+.4f}")
    print(f"  t = {comp_data['t_statistic']:.3f}, p = {comp_data['p_value']:.4f}")
    print(f"  Cohen's d = {comp_data['cohens_d']:.3f} ({comp_data['effect_size']})")
    print(f"  95% CI of difference: [{comp_data['bootstrap_ci_95'][0]:.4f}, {comp_data['bootstrap_ci_95'][1]:.4f}]")
    print(f"  Winner: {comp_data['winner']} ({comp_data['advantage_percent']:.1f}% better)")
    
    if comp_data['significant_at_01']:
        print(f"  *** HIGHLY SIGNIFICANT (p < 0.01) ***")
    elif comp_data['significant_at_05']:
        print(f"  ** SIGNIFICANT (p < 0.05) **")
    else:
        print(f"  Not statistically significant")

# Summary
print("\n" + "="*80)
print("SUMMARY")
print("="*80)

models_sorted = sorted(
    all_results['model_results'].items(),
    key=lambda x: x[1]['mean_loss']
)

print(f"\nRanking (by mean loss, lower is better):")
for rank, (model_path, results) in enumerate(models_sorted, 1):
    model_name = model_path.split('/')[-1]
    loss = results['mean_loss']
    ci = (results['ci_95_lower'], results['ci_95_upper'])
    ppl = results['perplexity']
    
    print(f"{rank}. {model_name}")
    print(f"   Loss: {loss:.4f} (95% CI: [{ci[0]:.4f}, {ci[1]:.4f}])")
    print(f"   Perplexity: {ppl:.2f}")

# Save
print(f"\nSaving results to: {_output}")
with open(_output, 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

# Visualizations
if _visualize:
    print("\nGenerating visualizations...")
    generate_visualizations(all_results, Path(_output).parent)

print("\n" + "="*80)
print("COMPLETE")
print("="*80)
print(f"\n✓ Evaluated {len(all_results['model_results'])} models")
print(f"✓ Statistical tests completed")
print(f"✓ Results saved to {_output}")

if _visualize:
    print(f"✓ Visualizations saved")

In [ ]:
Path(_output).parent

In [ ]:
all_results